# At times, code will be reaped from the project

### Below we have the manual implementation of the Gravity theory of trade

Used in 02_EDA in section 7.

In [ ]:
# In an attempt to synthesise the missing data in the dataframe, we can use the Gravity formula of trade.
# For more notes, see notes_for_work.md (03/04/2026)
#
# Question: Using pure raw gdp_o and gdp_d leads to very unbalanced synthetic data. Might be worth trying gdpcap_ppp_o


df = ecowas_df_full.copy()

# So to train our model, we must remove all NaNs. Don't want it to learn from what's missing, obviously.
train_df = df.dropna(subset=["tradeflow_baci", "gdpcap_ppp_o", "gdpcap_ppp_d", "dist", "pop_o", "pop_d"])

# Logs for variables included in PPML
train_df["ln_gdp_o"] = np.log(train_df["gdpcap_ppp_o"])
train_df["ln_gdp_d"] = np.log(train_df["gdpcap_ppp_d"])
train_df["ln_dist"]  = np.log(train_df["dist"])
train_df["ln_pop_o"] = np.log(train_df["pop_o"])
train_df["ln_pop_d"] = np.log(train_df["pop_d"])


# We want to include fixed effects. We use the pyfixest library:
# Since we currently do not have fixed effects, this is just a standard Pooled Poisson Pseudo-Maximum Likelihood regression (PPML)
res = fepois(
    fml="tradeflow_baci ~ ln_gdp_o + ln_gdp_d + ln_dist + sibling + ln_pop_o + ln_pop_d",        # Here we can add Fixed Effects by adding a | after the coefficients
    data=train_df,
    )

print(res.summary())



#### BELOW WE HAVE MANUAL IMPLEMENTATION OF GRAVITY TRADE MODEL
# Not necessary, since we have an automatic predict

# Here we can extract the coefficients. 
coefs = res.coef()

beta1 = coefs["ln_gdp_o"]
beta2 = coefs["ln_gdp_d"]
beta3 = -coefs["ln_dist"]  # negative because distance reduces trade

# Below we save the effects of the fixed effects in a dict
#fixef = res.fixef()

# We also need to figure out which fixed effects to include as part of this synthesising of data. 
# These fixed effects should make the synthetic data more precise, by adding other variables to map some of the complexity of the involved countries 
# 
numerator   = train_df["tradeflow_baci"].mean()

denominator = ((train_df["gdp_o"]**beta1) *
               (train_df["gdp_d"]**beta2) /
               (train_df["dist"]**beta3)).mean()

G = numerator / denominator
print(f"Our scaling constant G is: {G}")

df["ln_gdp_o"] = np.log(df["gdp_o"])
df["ln_gdp_d"] = np.log(df["gdp_d"])
df["ln_dist"]  = np.log(df["dist"])

df["synthetic_trade"] = (
    G *
    (df["gdp_o"] ** beta1) *
    (df["gdp_d"] ** beta2) /
    (df["dist"]  ** beta3)
)

# Optional: add noise (gamma noise works well for multiplicative models)
noise = np.random.gamma(shape=1.0, scale=0.15, size=len(df))
df["synthetic_trade_noisy"] = df["synthetic_trade"] * (1 + noise)




### Below we have an attempt to use multiple variables to synthesise the baci trade data (so all tradeflows are included)

Used in 03_synthetic_data in section 1.

The model we would ideally use for synthetic data (In this case tradeflow_baci):

E[BACI | X] = exp(
    β₀ + β₁ ln GDP_o + β₂ ln GDP_d + β₃ ln dist + β₄ contig + γ₁ ln Comtrade_o + γ₂ ln Comtrade_d + γ₃ ln IMF_o + γ₄ ln IMF_d
)

We would only keep the y_variables where they are present


In [ ]:
def safe_poisson_draw(lam, threshold=5e5):
    """
    Draw from Poisson when lambda is small,
    use Normal approximation when lambda is large.

    Added due to errors with values that are too high in the augmented values

    """
    lam = np.asarray(lam, dtype=float)
    out = np.empty_like(lam)

    small = lam <= threshold
    large = lam > threshold

    # Exact Poisson where safe
    out[small] = np.random.poisson(lam[small])

    # Normal approximation where Poisson breaks
    out[large] = np.maximum(
        0,
        np.random.normal(
            loc=lam[large],
            scale=np.sqrt(lam[large])
        )
    )

    return out


# We take it step by step. First we want to create a baseline model (that is, using just the β's from above)
# We want it to work for ALL countries
df_work = df.copy()

# We ensure positive values for the gravity variables
gravity_mask = (
    (df_work["gdp_o"] > 0) &
    (df_work["gdp_d"] > 0) &
    (df_work["distw_arithmetic"] > 0)
)

df_work = df_work.loc[gravity_mask].copy()

df_work["ln_gdp_o"] = np.log(df_work["gdp_o"])
df_work["ln_gdp_d"] = np.log(df_work["gdp_d"])
df_work["ln_distw_arithmetic"] = np.log(df_work["distw_arithmetic"])

mask_observations = df_work["tradeflow_baci"].notna()

X_base = sm.add_constant(df_work.loc[mask_observations, ["ln_gdp_o", "ln_gdp_d", "ln_distw_arithmetic", "contig"]])
y_base = df_work.loc[mask_observations, "tradeflow_baci"]

ppml_base = sm.GLM(
    y_base,
    X_base,
    family=sm.families.Poisson()
).fit(cov_type="HC1")

# Then we can predict for all base rows that we have
X_all_base = sm.add_constant(df_work[["ln_gdp_o", "ln_gdp_d", "ln_distw_arithmetic", "contig"]])

mu_base_hat = ppml_base.predict(X_all_base)
df_work["baci_mu_base"] = mu_base_hat



# Now we can add the auxiliary variables
aux_vars = [
    "tradeflow_comtrade_o",
    "tradeflow_comtrade_d",
    "tradeflow_imf_o",
    "tradeflow_imf_d"
]

for v in aux_vars:
    df_work[f"ln_{v}"] = np.where(
        df_work[v] > 0,
        np.log(df_work[v]),
        np.nan
    )


# We have an error - if any of the auxiliary variables have a single positive line but the rest are missing, we will get a line that contains NaNs or infinite values that break the statsmodel
# -> exog can not contain any NaNs or inf, or it breaks!
# Let's first define our augmented variables - Then we update the mask below so that any missing values fails the entire row
aug_vars = ["ln_gdp_o", "ln_gdp_d", "ln_distw_arithmetic", "contig"] + [
    f"ln_{v}" for v in aux_vars
]

# All right! Time to look at the augmented model (where applicable)
mask_aug = (
    df_work["tradeflow_baci"].notna() &
    #df_work[[f"ln_{v}" for v in aux_vars]].notna().any(axis=1)       This is the problematic line. This adds rows, even if there are some unusable values
    np.isfinite(df_work[aug_vars]).all(axis=1)
)



'''
X_aug = sm.add_constant(
    df_work.loc[mask_aug,
                ["ln_gdp_o", "ln_gdp_d", "ln_distw_arithmetic", "contig"] + [f"ln_{v}" for v in aux_vars]
                ]
)
'''
X_aug = sm.add_constant(df_work.loc[mask_aug, aug_vars])

y_aug = df_work.loc[mask_aug, "tradeflow_baci"]

ppml_aug = sm.GLM(
    y_aug,
    X_aug,
    family=sm.families.Poisson()
).fit(cov_type="HC1")

# We predict an augmented mean 
X_all_aug = sm.add_constant(
    df_work[["ln_gdp_o", "ln_gdp_d", "ln_distw_arithmetic", "contig"] + [f"ln_{v}" for v in aux_vars]]
)

mu_aug_hat = ppml_aug.predict(X_all_aug)
# we can save it as a column to use:
df_work["baci_mu_aug"] = mu_aug_hat

# We need to predict which prediction to use (so basically, augmented when possible, otherwise use the baseline)
has_any_aux = df_work[[f"ln_{v}" for v in aux_vars]].notna().any(axis=1)
has_all_aux = np.isfinite(df_work[[f"ln_{v}" for v in aux_vars]]).all(axis=1)

df_work["baci_mu_aug_final"] = np.where(
    has_any_aux,
    df_work["baci_mu_aug"],
    df_work["baci_mu_base"]
)

# The values got too big for PPML (but only slightly), so we need to implement a cap
# To inspect the values, you can see here: df_work["baci_mu_aug_final"].describe(percentiles=[0.9, 0.95, 0.99, 0.999])

cap = df_work["baci_mu_aug_final"].quantile(0.99)
df_work["baci_mu_aug_capped"] = np.clip(
    df_work["baci_mu_aug_final"],
    a_min = 0,
    a_max = cap
)

# Now we can synthesise it once and for all!
missing_baci = df_work["tradeflow_baci"].isna()

df_work.loc[missing_baci, "tradeflow_baci_synth"] = safe_poisson_draw(
    df_work.loc[missing_baci, "baci_mu_aug_capped"].values
)

df_work.loc[~missing_baci, "tradeflow_baci_synth"] = (
    df_work.loc[~missing_baci, "tradeflow_baci"]
)

df_work["baci_source"] = "observed"

df_work.loc[missing_baci & has_any_aux, "baci_source"] = "imputed_gravity+trade"
df_work.loc[missing_baci & ~has_any_aux, "baci_source"] = "imputed_gravity_only"


In [ ]:
#df must be defined

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score


# ==============================================================================
# 0. KONFIGURATION — justér lag-vinduet her
#    LAG = 1  →  konflikter fra t-1 bruges til at forudsige handel i t
#    LAG = 2  →  konflikter fra t-2 bruges til at forudsige handel i t
#    osv.
# ==============================================================================
LAG = 1  # <-- ændre denne værdi for at teste forskellige lag-vinduer

print("=" * 80)
print(f"   GRAVITY CONFLICT MODEL — LAG VINDUE: {LAG} ÅR")
print("=" * 80)


# ==============================================================================
# 1. DEFINER KONFLIKT-FEATURES DYNAMISK VHA. PRÆFIKS
# ==============================================================================
disorder_cols    = [c for c in df.columns if str(c).startswith('disorder_')]
event_cols       = [c for c in df.columns if str(c).startswith('event_')]
perpetrator_cols = [c for c in df.columns if str(c).startswith('perpetrator_')]
target_cols      = [c for c in df.columns if str(c).startswith('target_')]

all_acled_cols = disorder_cols + event_cols + perpetrator_cols + target_cols

print(f"\n✅ Konflikt-features fundet:")
print(f"   disorder_*    : {len(disorder_cols):>3} kolonner")
print(f"   event_*       : {len(event_cols):>3} kolonner")
print(f"   perpetrator_* : {len(perpetrator_cols):>3} kolonner")
print(f"   target_*      : {len(target_cols):>3} kolonner")
print(f"   Total         : {len(all_acled_cols):>3} kolonner\n")

df_mod = df.copy()


# ==============================================================================
# 2. VALIDÉR AT TIDSSERIEN ER KONTINUERT PER DYAD
#    Shift(n) på hullede tidsserier kobler forkerte år sammen —
#    f.eks. lag=1 på en tidsserie med hul i 2011 kobler 2012-handel
#    med 2010-konflikter i stedet for 2011-konflikter.
# ==============================================================================
df_mod = df_mod.sort_values(by=["dyad", "year"])

year_gaps      = df_mod.groupby("dyad")["year"].diff()
n_gaps         = (year_gaps.dropna() > 1).sum()
n_dyads_w_gaps = df_mod[year_gaps > 1]["dyad"].nunique()

if n_gaps > 0:
    print(
        f"⚠️  Advarsel: {n_gaps} hullede årsovergange fundet i {n_dyads_w_gaps} dyader.\n"
        f"   Lag({LAG}) kobler muligvis forkerte år. Overvej at udfylde hullerne\n"
        f"   med NaN-rækker eller filtrere de berørte dyader fra.\n"
    )
else:
    print(f"✅ Tidsserier er kontinuerte — lag({LAG}) er korrekt.\n")


# ==============================================================================
# 3. SKAB DYNAMISK LAG FOR KONFLIKTDATA PER DYAD
#    shift(LAG) skubber konfliktvariablene LAG år fremad inden for hver dyad.
#    De første LAG år per dyad får NaN og droppes per model inde i funktionerne.
# ==============================================================================
df_mod[all_acled_cols] = df_mod.groupby("dyad")[all_acled_cols].shift(LAG)

print(f"✅ Konfliktdata lagget {LAG} år per dyad (shift={LAG}).\n")


# ==============================================================================
# 4. KLARGØR GRAVITY FEATURES OG TARGET (log-transformationer)
# ==============================================================================
df_mod["baci_log_trade"] = np.log(df_mod["tradeflow_baci"].replace(0, np.nan))
df_mod["log_gdp_o"]      = np.log(df_mod["gdp_o"])
df_mod["log_gdp_d"]      = np.log(df_mod["gdp_d"])
df_mod["log_distw"]      = np.log(df_mod["distw_arithmetic"])

if np.isinf(df_mod["log_distw"]).any():
    n_inf = np.isinf(df_mod["log_distw"]).sum()
    print(f"⚠️  Advarsel: log_distw indeholder {n_inf} -inf værdier (distw_arithmetic = 0).")

gravity_cols = ["log_gdp_o", "log_gdp_d", "log_distw"]

df_mod = df_mod.dropna(subset=gravity_cols + ["baci_log_trade"])

# Aggregeret konfliktintensitet med log1p-transformation (Santos Silva & Tenreyro)
# log1p bruges fordi der kan være nul-hændelser: log1p(0) = 0 (undgår -inf)
df_mod["conflict_total"]     = df_mod[event_cols].sum(axis=1)
df_mod["log_conflict_total"] = np.log1p(df_mod["conflict_total"])

print("✅ Log-transformationer anvendt:")
print(f"   baci_log_trade     = log(tradeflow_baci)         — nuller erstattet med NaN")
print(f"   log_gdp_o          = log(gdp_o)")
print(f"   log_gdp_d          = log(gdp_d)")
print(f"   log_distw          = log(distw_arithmetic)")
print(f"   log_conflict_total = log1p(sum af event_*-kolonner)\n")


# ==============================================================================
# 5A. OLS BASELINE FUNKTION
#     Target    : baci_log_trade (log-transformeret handel)
#     Estimator : OLS med clustered SE på dyad-niveau
#     RMSE/R²   : log-skala
# ==============================================================================
def run_ols_model(conflict_features: list[str], model_name: str) -> dict:

    print(f"\n{'='*80}")
    print(f"   {model_name}  [OLS, lag={LAG}]")
    print(f"{'='*80}\n")

    features = gravity_cols + conflict_features
    df_clean = df_mod.dropna(subset=features)

    train_df = df_clean[df_clean["year"] <= 2016]
    test_df  = df_clean[df_clean["year"] > 2016]

    X_train = train_df[features]
    y_train = train_df["baci_log_trade"]
    X_test  = test_df[features]
    y_test  = test_df["baci_log_trade"]

    n_features = len(features)
    print(f"Features ({n_features} total):")
    for f in features:
        print(f"   + {f}")
    print(f"\nObservationer — træning: {len(train_df):,}  |  test: {len(test_df):,}")
    print(f"År i træning  : {train_df['year'].min()} – {train_df['year'].max()}")
    print(f"År i test     : {test_df['year'].min()}  – {test_df['year'].max()}\n")

    # Sklearn til out-of-sample evaluering
    sk_model = LinearRegression()
    sk_model.fit(X_train, y_train)
    y_pred = sk_model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"OLS →  Out-of-sample RMSE (log-skala) : {rmse:.4f}")
    print(f"OLS →  Out-of-sample R²   (log-skala) : {r2:.4f}")
    print(f"       (fortolkning: modellen forklarer {r2*100:.1f}% af variationen i log-handel\n"
          f"        på data den ikke er trænet på)\n")

    # Statsmodels til inferens med clustered SE
    X_train_sm = sm.add_constant(X_train)
    ols = sm.OLS(y_train, X_train_sm).fit(
        cov_type="cluster",
        cov_kwds={"groups": train_df["dyad"]}
    )
    print(ols.summary())

    # Udskriv signifikante koefficienter (p < 0.05) som opsummering
    sig = ols.pvalues[ols.pvalues < 0.05].drop("const", errors="ignore")
    if len(sig) > 0:
        print(f"\n📌 Signifikante konfliktkoefficienter (p < 0.05):")
        for var in sig.index:
            if var in conflict_features:
                direction = "↑ positiv" if ols.params[var] > 0 else "↓ negativ"
                print(f"   {var:<45} coef={ols.params[var]:+.4f}  p={ols.pvalues[var]:.3f}  {direction}")
    else:
        print(f"\n📌 Ingen konfliktkoefficienter er signifikante på p < 0.05")

    return {
        "model_name"  : model_name,
        "model_type"  : "OLS",
        "lag"         : LAG,
        "n_train"     : len(train_df),
        "n_test"      : len(test_df),
        "rmse"        : rmse,
        "r2"          : r2,
        "ols_results" : ols,
    }


# ==============================================================================
# 5B. GPML FUNKTION (Gamma Pseudo Maximum Likelihood)
#     Santos Silva & Tenreyro (2006) — løser Jensen's inequality bias i OLS
#     og håndterer nul-handel uden at droppe observationer.
#
#     Target    : tradeflow_baci i niveauer (ikke log — log-link anvendes internt)
#     Estimator : Gamma GLM med log-link + clustered SE på dyad-niveau
#     RMSE/R²   : rapporteres på både log- og niveau-skala
# ==============================================================================
def run_gpml_model(conflict_features: list[str], model_name: str) -> dict:

    print(f"\n{'='*80}")
    print(f"   {model_name}  [GPML, lag={LAG}]")
    print(f"{'='*80}\n")

    features = gravity_cols + conflict_features

    # GPML inkluderer nul-handel — vi dropper kun NaN, ikke nuller
    df_clean = df_mod.dropna(subset=features + ["tradeflow_baci"])

    train_df = df_clean[df_clean["year"] <= 2016]
    test_df  = df_clean[df_clean["year"] > 2016]

    # Target er handel i niveauer — log-link håndteres internt af GLM
    X_train = sm.add_constant(train_df[features])
    y_train = train_df["tradeflow_baci"]
    X_test  = sm.add_constant(test_df[features])
    y_test  = test_df["tradeflow_baci"]

    n_features = len(features)
    n_zeros    = (y_train == 0).sum()

    print(f"Features ({n_features} total):")
    for f in features:
        print(f"   + {f}")
    print(f"\nObservationer — træning: {len(train_df):,}  |  test: {len(test_df):,}")
    print(f"År i træning  : {train_df['year'].min()} – {train_df['year'].max()}")
    print(f"År i test     : {test_df['year'].min()}  – {test_df['year'].max()}")
    print(f"Nul-handel i træning: {n_zeros:,} rækker (inkluderet i GPML, ville være droppet i OLS)\n")

    gpml = sm.GLM(
        y_train,
        X_train,
        family=sm.families.Gamma(link=sm.families.links.Log())
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": train_df["dyad"]},
        maxiter=200
    )
    print(gpml.summary())

    # --- Niveau-skala evaluering ---
    y_pred_levels = gpml.predict(X_test)
    rmse_levels   = root_mean_squared_error(y_test, y_pred_levels)
    r2_levels     = r2_score(y_test, y_pred_levels)

    # --- Log-skala evaluering (sammenlignelig med OLS) ---
    # Clip predictions til minimum 1 for at undgå log(0)
    y_pred_log = np.log(np.clip(y_pred_levels, 1, None))
    y_test_log = np.log(y_test.replace(0, np.nan)).dropna()
    y_pred_log = y_pred_log[y_test_log.index]  # align index efter dropna

    rmse_log = root_mean_squared_error(y_test_log, y_pred_log)
    r2_log   = r2_score(y_test_log, y_pred_log)

    print(f"\nGPML →  Out-of-sample RMSE (log-skala)    : {rmse_log:.4f}  ← sammenlignelig med OLS")
    print(f"GPML →  Out-of-sample R²   (log-skala)    : {r2_log:.4f}  ← sammenlignelig med OLS")
    print(f"        (fortolkning: modellen forklarer {r2_log*100:.1f}% af variationen i log-handel\n"
          f"         på data den ikke er trænet på)\n")
    print(f"GPML →  Out-of-sample RMSE (niveau-skala) : {rmse_levels:,.2f}  (ikke sammenlignelig med OLS)")
    print(f"GPML →  Out-of-sample R²   (niveau-skala) : {r2_levels:.4f}  (ikke sammenlignelig med OLS)")

    # Signifikante koefficienter
    sig = gpml.pvalues[gpml.pvalues < 0.05].drop("const", errors="ignore")
    if len(sig) > 0:
        print(f"\n📌 Signifikante konfliktkoefficienter (p < 0.05):")
        for var in sig.index:
            if var in conflict_features:
                direction = "↑ positiv" if gpml.params[var] > 0 else "↓ negativ"
                print(f"   {var:<45} coef={gpml.params[var]:+.4f}  p={gpml.pvalues[var]:.3f}  {direction}")
    else:
        print(f"\n📌 Ingen konfliktkoefficienter er signifikante på p < 0.05")

    return {
        "model_name"  : model_name,
        "model_type"  : "GPML",
        "lag"         : LAG,
        "n_train"     : len(train_df),
        "n_test"      : len(test_df),
        "rmse"        : rmse_log,
        "r2"          : r2_log,
        "rmse_levels" : rmse_levels,
        "r2_levels"   : r2_levels,
        "gpml_results": gpml,
    }


# ==============================================================================
# 6. KØR ALLE MODELLER
#    Model 1-4 : OLS med subtype-konfliktfeatures
#    Model 5   : OLS med aggregeret log-konfliktintensitet
#    Model 6   : GPML med aggregeret log-konfliktintensitet
# ==============================================================================
print(f"\n{'='*80}")
print(f"   KØRER MODELLER MED LAG = {LAG} ÅR")
print(f"{'='*80}")

ols_specs = [
    (disorder_cols,          "1. OLS — Disorder"),
    (event_cols,             "2. OLS — Event"),
    (perpetrator_cols,       "3. OLS — Perpetrator"),
    (target_cols,            "4. OLS — Target"),
    (["log_conflict_total"], "5. OLS — Log Conflict Intensity (aggregeret)"),
]

gpml_specs = [
    (["log_conflict_total"], "6. GPML — Log Conflict Intensity (aggregeret)"),
]

ols_results  = [run_ols_model(cols, name)  for cols, name in ols_specs]
gpml_results = [run_gpml_model(cols, name) for cols, name in gpml_specs]

all_results = ols_results + gpml_results


# ==============================================================================
# 7. SAMMENLIGNINGSTABEL PÅ TVÆRS AF ALLE MODELLER
#    RMSE og R² er på log-skala for alle modeller — direkte sammenlignelige.
# ==============================================================================
comparison = pd.DataFrame(
    [
        {
            "Model"      : r["model_name"],
            "Type"       : r["model_type"],
            "Lag"        : r["lag"],
            "N train"    : r["n_train"],
            "N test"     : r["n_test"],
            "RMSE (log)" : round(r["rmse"], 4),
            "R²   (log)" : round(r["r2"],   4),
        }
        for r in all_results
    ]
).set_index("Model")

print("\n\n" + "=" * 80)
print(f"   SAMMENLIGNING — OUT-OF-SAMPLE PERFORMANCE (år > 2016, log-skala, lag={LAG})")
print(f"   Alle RMSE/R² er på log-skala — direkte sammenlignelige på tværs af modeller")
print("=" * 80)
print(comparison.to_string())
print("=" * 80)

# Bedste model
best = comparison["RMSE (log)"].idxmin()
print(f"\n🏆 Bedste model (lavest RMSE): {best}")
print(f"   RMSE (log) = {comparison.loc[best, 'RMSE (log)']}")
print(f"   R²   (log) = {comparison.loc[best, 'R²   (log)']}\n")

# GPML niveau-skala metrics som reference
gpml_ref = pd.DataFrame(
    [
        {
            "Model"         : r["model_name"],
            "RMSE (niveau)" : round(r["rmse_levels"], 2),
            "R²   (niveau)" : round(r["r2_levels"],   4),
        }
        for r in gpml_results
    ]
).set_index("Model")

print("GPML — NIVEAU-SKALA METRICS (reference, ikke sammenlignelige med OLS):")
print(gpml_ref.to_string())
print()

   GRAVITY CONFLICT MODEL — LAG VINDUE: 1 ÅR


NameError: name 'df' is not defined

In [2]:
def run_nb_model(conflict_features: list[str], model_name: str) -> dict:
    print(f"\n{'='*80}")
    print(f"   {model_name}")
    print(f"{'='*80}\n")

    features = gravity_cols + conflict_features

    # NB kræver positive, ikke-NaN counts — brug utransformeret handel
    df_clean = df_mod.dropna(subset=features + ["tradeflow_baci"])
    df_clean = df_clean[df_clean["tradeflow_baci"] > 0]

    train_df = df_clean[df_clean["year"] <= 2016]
    test_df  = df_clean[df_clean["year"] > 2016]

    # NB target: handel i niveauer (ikke log)
    X_train = sm.add_constant(train_df[features])
    y_train = train_df["tradeflow_baci"]
    X_test  = sm.add_constant(test_df[features])
    y_test  = test_df["tradeflow_baci"]

    print(f"Observationer — træning: {len(train_df):,}  |  test: {len(test_df):,}\n")

    nb_model = sm.NegativeBinomial(y_train, X_train).fit(
        cov_type="cluster",
        cov_kwds={"groups": train_df["dyad"]},
        maxiter=200,
        disp=False
    )
    print(nb_model.summary())

    # --- Evaluering på niveau-skala ---
    y_pred_levels = nb_model.predict(X_test)
    rmse_levels   = root_mean_squared_error(y_test, y_pred_levels)
    r2_levels     = r2_score(y_test, y_pred_levels)

    print(f"\nNB  →  Out-of-sample RMSE (niveau-skala): {rmse_levels:.4f}")
    print(f"NB  →  Out-of-sample R²   (niveau-skala): {r2_levels:.4f}")

    # --- Evaluering på log-skala (sammenlignelig med OLS-modellerne) ---
    # Log-transformer både predictions og faktiske værdier
    # Clip predictions til minimum 1 for at undgå log(0)
    y_pred_log = np.log(np.clip(y_pred_levels, 1, None))
    y_test_log = np.log(y_test)

    rmse_log = root_mean_squared_error(y_test_log, y_pred_log)
    r2_log   = r2_score(y_test_log, y_pred_log)

    print(f"NB  →  Out-of-sample RMSE (log-skala):    {rmse_log:.4f}  ← sammenlignelig med OLS")
    print(f"NB  →  Out-of-sample R²   (log-skala):    {r2_log:.4f}  ← sammenlignelig med OLS")

    return {
        "model_name"  : model_name,
        "model_type"  : "NB",
        "n_train"     : len(train_df),
        "n_test"      : len(test_df),
        "rmse"        : rmse_log,       # log-skala bruges i sammenligningstabellen
        "r2"          : r2_log,         # log-skala bruges i sammenligningstabellen
        "rmse_levels" : rmse_levels,
        "r2_levels"   : r2_levels,
        "nb_results"  : nb_model,
    }
